In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

# Reload the raw, committed dataset (single source of truth)
df = pd.read_csv("titanic.csv")
print("shape:", df.shape)
df.head(3)

NUMERIC_FEATURES = ["pclass", "age", "sibsp", "parch", "fare"]
CATEGORICAL_FEATURES = ["sex", "embarked"]
TARGET = "survived"

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Overall survival rate:      ", round(y.mean(), 4))
print("Train survival rate:        ", round(y_train.mean(), 4))
print("Test survival rate:         ", round(y_test.mean(), 4))
print(f"Train size: {len(X_train)}   Test size: {len(X_test)}")

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_FEATURES),
    ])

_prep_check = make_preprocessor().fit(X_train)
print("Transformed feature count:", _prep_check.transform(X_test).shape[1])
print("Feature names:", list(_prep_check.get_feature_names_out()))

classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

fitted_pipelines = {}
for name, clf in classifiers.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
print("Trained:", list(fitted_pipelines.keys()))

feature_names = fitted_pipelines["DecisionTree"].named_steps["prep"].get_feature_names_out()
tree_model = fitted_pipelines["DecisionTree"].named_steps["clf"]

plt.figure(figsize=(16, 8))
plot_tree(
    tree_model, max_depth=3, feature_names=feature_names,
    class_names=["Not Survived", "Survived"], filled=True, fontsize=8, rounded=True,
)
plt.title("Decision Tree (rendered to depth 3 for legibility; full tree is deeper and is what's scored)")
plt.tight_layout()
plt.savefig("charts/09_decision_tree.png", dpi=110)
plt.show()

metrics_rows = []
fig_cm, axes_cm = plt.subplots(1, 3, figsize=(15, 4.5))
fig_roc, ax_roc = plt.subplots(figsize=(6, 5.5))

for (name, pipe), ax_cm in zip(fitted_pipelines.items(), axes_cm):
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    metrics_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })

    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                            display_labels=["Not Survived", "Survived"]).plot(ax=ax_cm, colorbar=False)
    ax_cm.set_title(name)

    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4, label="chance")
ax_roc.set_title("ROC curves -- all 3 classifiers")
ax_roc.legend()
fig_cm.tight_layout(); fig_cm.savefig("charts/10_confusion_matrices.png", dpi=110)
fig_roc.tight_layout(); fig_roc.savefig("charts/11_roc_curves.png", dpi=110)
plt.show()

classifier_comparison = pd.DataFrame(metrics_rows).set_index("model").round(4)
classifier_comparison

print("Class balance (whole dataset):")
print(y.value_counts(normalize=True).rename("proportion"))

imbalance_rows = []

pipe_base = Pipeline([("prep", make_preprocessor()), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
pipe_base.fit(X_train, y_train)
pred = pipe_base.predict(X_test)
imbalance_rows.append(("baseline", precision_score(y_test, pred), recall_score(y_test, pred), f1_score(y_test, pred)))

pipe_bal = Pipeline([("prep", make_preprocessor()),
                     ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"))])
pipe_bal.fit(X_train, y_train)
pred = pipe_bal.predict(X_test)
imbalance_rows.append(("class_weight_balanced", precision_score(y_test, pred), recall_score(y_test, pred), f1_score(y_test, pred)))

prep_for_smote = make_preprocessor().fit(X_train)
X_train_enc = prep_for_smote.transform(X_train)
X_test_enc = prep_for_smote.transform(X_test)
X_train_sm, y_train_sm = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train_enc, y_train)
clf_sm = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf_sm.fit(X_train_sm, y_train_sm)
pred = clf_sm.predict(X_test_enc)
imbalance_rows.append(("smote", precision_score(y_test, pred), recall_score(y_test, pred), f1_score(y_test, pred)))

imbalance_comparison = pd.DataFrame(imbalance_rows, columns=["strategy", "precision", "recall", "f1"]).set_index("strategy").round(4)
imbalance_comparison

rf_pipe = Pipeline([
    ("prep", make_preprocessor()),
    ("clf", RandomForestClassifier(oob_score=True, bootstrap=True, random_state=RANDOM_STATE)),
])

param_grid = {
    "clf__n_estimators": [100, 200, 400],
    "clf__max_depth": [4, 8, None],
    "clf__max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

best_rf_pipeline = grid.best_estimator_
oob = best_rf_pipeline.named_steps["clf"].oob_score_

print("Best params:        ", grid.best_params_)
print("Best CV f1:          ", round(grid.best_score_, 4))
print("OOB score (refit on full training data with best params):", round(oob, 4))

tuned_pred = best_rf_pipeline.predict(X_test)
tuned_proba = best_rf_pipeline.predict_proba(X_test)[:, 1]
tuned_metrics = {
    "accuracy": accuracy_score(y_test, tuned_pred),
    "precision": precision_score(y_test, tuned_pred),
    "recall": recall_score(y_test, tuned_pred),
    "f1": f1_score(y_test, tuned_pred),
    "roc_auc": roc_auc_score(y_test, tuned_proba),
}
print("
Tuned RandomForest -- held-out test metrics:")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v:.4f}")

REG_NUMERIC = ["pclass", "age", "sibsp", "parch"]
REG_CATEGORICAL = ["sex", "embarked"]

Xr = df[REG_NUMERIC + REG_CATEGORICAL]
yr = df["fare"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=RANDOM_STATE)

reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), REG_NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), REG_CATEGORICAL),
])
reg_pipeline = Pipeline([("prep", reg_preprocessor), ("reg", LinearRegression())])
reg_pipeline.fit(Xr_train, yr_train)

yr_pred = reg_pipeline.predict(Xr_test)
mae = mean_absolute_error(yr_test, yr_pred)
rmse = mean_squared_error(yr_test, yr_pred) ** 0.5
r2 = r2_score(yr_test, yr_pred)

n = len(yr_test)
p = reg_pipeline.named_steps["prep"].transform(Xr_test).shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

regression_metrics = pd.Series({"MAE": mae, "RMSE": rmse, "R2": r2, "Adjusted_R2": adj_r2}).round(4)
print(regression_metrics)

residuals = yr_test - yr_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(yr_pred, residuals, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted fare"); axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs. fitted values")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.savefig("charts/12_regression_residuals.png", dpi=110)
plt.show()

print("Classifier comparison (Task 9/10 models):")
display_classifiers = classifier_comparison.copy()
display_classifiers.loc["RandomForest (tuned, Task 12)"] = tuned_metrics
display_classifiers = display_classifiers.round(4)
display_classifiers

print("Regression comparison (Task 13 model):")
regression_comparison = regression_metrics.to_frame(name="LinearRegression_fare").T
regression_comparison

joblib.dump(best_rf_pipeline, "model_pipeline.joblib")
print("Saved -> model_pipeline.joblib")

reloaded_pipeline = joblib.load("model_pipeline.joblib")

raw_sample = pd.DataFrame([
    {"pclass": 1, "age": 29.0, "sibsp": 0, "parch": 0, "fare": 211.34, "sex": "female", "embarked": "S"},
    {"pclass": 3, "age": 22.0, "sibsp": 1, "parch": 0, "fare": 7.25, "sex": "male", "embarked": "S"},
])
reloaded_pred = reloaded_pipeline.predict(raw_sample)
reloaded_proba = reloaded_pipeline.predict_proba(raw_sample)[:, 1]

print("
Prediction on brand-new raw rows (no manual preprocessing applied by us):")
for i in range(len(raw_sample)):
    print(f"  row {i}: predicted survived={reloaded_pred[i]}  P(survived)={reloaded_proba[i]:.4f}")

assert list(reloaded_pred) == list(best_rf_pipeline.predict(raw_sample)), "reloaded pipeline should match the original"
print("
Reloaded pipeline's predictions match the original in-memory pipeline. Round-trip verified.")
